<a href="https://colab.research.google.com/github/ajakhmol/CNN/blob/main/SummaryCreater.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# prompt: write a iterator on the criteria_list list and pass the value of the list in the generate_response function
import pandas as pd
from IPython.display import display, HTML
import os, json, ast
import openai
from tenacity import retry, wait_random_exponential, stop_after_attempt
from google.colab import drive
# Set the display width to control the output width
# pd.set_option('display.width', 500)
from io import StringIO

from tenacity import retry, wait_random_exponential, stop_after_attempt

In [ ]:
# # criteria_list = [
#     "Scalability and Performance"
#     "Security and Compliance",
#     "User Experience and Adoption",
#     "Integration and Ecosystem",
#     "Administration and Governance",
#     "Support and Training",
#     "Disaster Recovery and Uptime",
#     "Customization and Flexibility",
#     "Cost Optimization",
#     "AI, LLM, and LCM Integration",
#     "Regulatory and Political Landscape",
#     "Salesforce Roadmap and Future Plan",
#     "Resource Reskilling",
#     "Data Migration and Management",
#     "Change Management and User Adoption",
#     "Vendor Management and Relationship"
#     "Project Management and Implementation",
#     "Innovation and Future-Proofing"
# # ]


In [ ]:

# Importing the necessary library for mounting Google Drive

# Mounting Google Drive to the Colab environment
drive.mount('/content/drive', force_remount=True)
# Read the API key from the text file and strip any leading or trailing whitespace
with open("/content/drive/My Drive/Gen_AI/OPENAI_API_Key.txt", "r") as f:
    api_key = f.read().strip()

# Set the API key for OpenAI
openai.api_key = api_key

Mounted at /content/drive


In [ ]:
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def generate_ai_response_subtopic(df_csv):
    """
    Generate a response using gpt-4o-mini ChatCompletion.  Returns a CSV string.
    """
    system_content="You are a helpful AI assistant in the Financial Technology consulting domain, specialized in providing accurate answers. Respond with ONLY a CSV string, with the first row being the header. Do not include any text before or after the CSV data."

    user_content= f"""
                  The evaluation focuses on U.S.-based financial institutions considering Salesforce Experience Cloud
                  as an alternative to their existing home-grown solution, which is built on Java, FTL, Oracle, and APIs.
                  Below is my DataFrame in CSV format:
                  {df_csv}

                  Please  give the definition of every Subcriterion 'Evaluation Subcriterion' in the dataframe.
                  definition must NOT more than 40 words.

                  """

    messages = [
        {"role": "system", "content": system_content },
        {"role": "user", "content": user_content },
    ]

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    content = response.choices[0].message.content
    return content # Return the raw CSV string

In [ ]:
def process_excel_file(file_path, file_name):
    try:
        xl = pd.ExcelFile(os.path.join(file_path, file_name))
        for sheet_name in xl.sheet_names:  # Iterate through all sheets
            print(f"Processing sheet: {sheet_name}")
            df = xl.parse(sheet_name)
            print("Original dataframe")
            display(df)

            if 'Definition' not in df.columns:
                df['Definition'] = ''

            # Correctly select columns using .loc[]
            df = df.loc[:, ['Evaluation Criterion', 'Evaluation Subcriterion', 'Definition']]

            # Remove duplicates based on all columns
            df = df.drop_duplicates(subset=['Evaluation Criterion', 'Evaluation Subcriterion'])

            df_csv = df.to_csv(index=False)
            csv_response = generate_ai_response_subtopic(df_csv)

            try:
                response_df = pd.read_csv(StringIO(csv_response))
                print("Response dataframe")
                display(response_df)
            except pd.errors.ParserError as e:
                print(f"Error parsing CSV response for sheet {sheet_name}: {e}")
                print(f"Raw response:\n{csv_response}")

            # ... (rest of your code)

            # Create a new Excel file to store the results
            output_file_path = os.path.join(file_path, 'processed_'+ file_name)
            try:
              with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                  df.to_excel(writer, sheet_name=sheet_name, index=False)
              print(f"Sheet '{sheet_name}' processed and saved to '{output_file_path}'")
            except FileNotFoundError:
              with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='w') as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
              print(f"Sheet '{sheet_name}' processed and saved to '{output_file_path}' - new file created")

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
    except Exception as e:
        print(f"An error occurred: {e}")




In [ ]:
file_path = "/content/drive/My Drive/Gen_AI/Out"
file_name = "All_In_One_primary.xlsx"
process_excel_file(file_path,file_name)

Processing sheet: Scalability and Performance
Original dataframe


,Evaluation Criterion,Importance Scale,Evaluation Subcriterion,Parameters of the Subcriterion,Score(0-5),Definition
0,Scalability and Performance,5,Concurrent user capacity,Maximum concurrent users (10000+),1,The ability of the system to efficiently handl...
1,Scalability and Performance,5,Concurrent user capacity,Annual growth rate (10%),1,Annual growth rate refers to the percentage in...
2,Scalability and Performance,5,Concurrent user capacity,Peak load response time,1,Peak load response time measures the system's ...
3,Scalability and Performance,5,Concurrent user capacity,System latency,1,System latency refers to the time delay experi...
4,Scalability and Performance,5,Concurrent user capacity,Resource allocation efficiency,5,Resource allocation efficiency measures the ef...
...,...,...,...,...,...,...
222,Scalability and Performance,4,Capacity planning for peak usage,Cost estimation for scaling,5,Cost estimation for scaling refers to the proj...
223,Scalability and Performance,4,Capacity planning for peak usage,Performance of database queries,5,The measure of a system's ability to handle in...
224,Scalability and Performance,4,Capacity planning for peak usage,API call limits,5,The maximum number of requests the system can ...
225,Scalability and Performance,4,Capacity planning for peak usage,User education and training,5,User education and training refer to the proce...


Error parsing CSV response for sheet Scalability and Performance: Error tokenizing data. C error: Expected 3 fields in line 4, saw 4

Raw response:
Evaluation Criterion,Evaluation Subcriterion,Definition
Scalability and Performance,Concurrent user capacity,Ability to handle multiple users simultaneously while ensuring optimal performance and quick response times during high load.
Scalability and Performance,Data storage and growth management,Management of current storage capacity and future data growth to ensure effective scalability.
Scalability and Performance,API throughput and optimization,Duration to process requests and return results, affecting user experience and system efficiency.
Scalability and Performance,Caching strategies,Method of storing cached data to optimize speed and memory efficiency.
Scalability and Performance,CDN,Time taken for data to travel from the CDN to the user, affecting content loading speed.
Scalability and Performance,Load balancing,Distribution of tra

,Evaluation Criterion,Importance Scale,Evaluation Subcriterion,Parameters of the Subcriterion,Score(0-5),Definition
0,Security and Compliance,5,Data encryption (at rest and in transit),"Encryption standards (AES, RSA)",1,The measure of effectiveness and robustness in...
1,Security and Compliance,5,Data encryption (at rest and in transit),Key management procedures,1,Key management procedures are the practices an...
2,Security and Compliance,5,Data encryption (at rest and in transit),"Compliance with regulations (GDPR,CCPA,GLBA)",1,Security and Compliance pertains to the measur...
3,Security and Compliance,5,Data encryption (at rest and in transit),Data access control,1,Measures and policies ensuring that only autho...
4,Security and Compliance,5,Data encryption (at rest and in transit),Performance impact assessments,1,Performance impact assessments evaluate how se...
...,...,...,...,...,...,...
226,Security and Compliance,5,Incident response plan,Budget allocation for incident response,5,Budget allocation for incident response refers...
227,Security and Compliance,5,Incident response plan,Business impact analysis processes,5,Business impact analysis processes assess pote...
228,Security and Compliance,5,Incident response plan,Stakeholder involvement during incidents,5,Stakeholder involvement during incidents refer...
229,Security and Compliance,5,Incident response plan,Frequency of incident plan updates,5,The frequency of incident plan updates refers ...


Response dataframe


...

Sheet 'Project Management' processed and saved to '/content/drive/My Drive/Gen_AI/Out/processed_All_In_One_primary.xlsx'
Processing sheet: Innovation and Future-Proofing
Original dataframe


,Evaluation Criterion,Importance Scale,Evaluation Subcriterion,Parameters of the Subcriterion,Score(0-5),Definition
0,Innovation and Future-Proofing,5,Staying current with Salesforce releases,Regular training sessions,5,The process of maintaining up-to-date knowledg...
1,Innovation and Future-Proofing,5,Staying current with Salesforce releases,Release notes review,5,Release notes review involves analyzing update...
2,Innovation and Future-Proofing,5,Staying current with Salesforce releases,Feature adoption metrics,5,Feature adoption metrics measure the rate and ...
3,Innovation and Future-Proofing,5,Staying current with Salesforce releases,Feedback loop from users,5,A process wherein users provide insights and s...
4,Innovation and Future-Proofing,5,Staying current with Salesforce releases,Integration testing,5,Integration testing assesses the functionality...
...,...,...,...,...,...,...
319,Innovation and Future-Proofing,5,Continuous improvement and optimization,Maintenance and support response times,5,Continuous improvement and optimization refers...
320,Innovation and Future-Proofing,5,Continuous improvement and optimization,Reduction in technical debt,5,A measure of ongoing enhancements in technolog...
321,Innovation and Future-Proofing,5,Continuous improvement and optimization,Continuous learning mechanisms,5,Continuous learning mechanisms refer to proces...
322,Innovation and Future-Proofing,5,Continuous improvement and optimization,System error rates,5,A measure of the frequency of system malfuncti...


Response dataframe


,Evaluation Criterion,Evaluation Subcriterion,Definition
0,Innovation and Future-Proofing,Staying current with Salesforce releases,The process of maintaining knowledge on Salesf...
1,Innovation and Future-Proofing,Staying current with Salesforce updates,Checks for compatibility with Salesforce updat...
2,Innovation and Future-Proofing,"Exploring emerging technologies (e.g., blockch...",Scalability refers to the platform's ability t...
3,Innovation and Future-Proofing,Impact of blockchain on transactions,Transaction speed measures how quickly transac...
4,Innovation and Future-Proofing,Web3 applications and finance,"Decentralization benefits improve security, tr..."
5,Innovation and Future-Proofing,AI and machine learning in finance,Predictive analytics uses data to forecast tre...
6,Innovation and Future-Proofing,Digital wallets and their integration,User experience assesses customer satisfaction...
7,Innovation and Future-Proofing,Integration of open APIs,Flexibility allows platforms to adapt and inte...
8,Innovation and Future-Proofing,Real-time data analytics,Accuracy of insights evaluates the reliability...
9,Innovation and Future-Proofing,Evaluating potential integrations with other p...,Existing API capabilities assess how interface...


Sheet 'Innovation and Future-Proofing' processed and saved to '/content/drive/My Drive/Gen_AI/Out/processed_All_In_One_primary.xlsx'
